# 02 — Generate manuscript model outputs

Clean publication version. Uses repository-relative paths and writes validated north-up GeoTIFFs.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import os

def find_project_root(start=None):
    """Find repository root from either repo root or analysis/."""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate publication repository root. "
        "Expected sibling 'src' and 'data' directories."
    )

project_root = find_project_root()
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from model_config import baseline_params, teredo_k, alpha_rate
from decay_model import (
    compute_total_decay_full,
    compute_steel_reference,
    build_rate_scale,
    time_to_threshold_map,
)

print("Project root:", project_root)


In [ ]:
data_dir = project_root / "data"
figure_data_dir = project_root / "outputs" / "figure_data"
figure_map_dir = project_root / "outputs" / "figure_maps"
driver_dir = project_root / "outputs" / "driver_maps"

for directory in (figure_data_dir, figure_map_dir, driver_dir):
    directory.mkdir(parents=True, exist_ok=True)

regional_file = data_dir / "model_inputs" / "publication_model_inputs.nc"
curve_file = data_dir / "gam_age_curve.csv"

if not regional_file.exists():
    raise FileNotFoundError(
        f"Missing publication model input: {regional_file}\n"
        "Run 00_prepare_publication_data.ipynb first."
    )

if not curve_file.exists():
    raise FileNotFoundError(
        f"Missing GAM curve: {curve_file}\n"
        "Run 01_calibration.ipynb first."
    )

ABMsubset = xr.open_dataset(regional_file, chunks="auto")
gam_curve = pd.read_csv(curve_file)

print("Regional dimensions:", dict(ABMsubset.sizes))
print("Regional variables:", len(ABMsubset.data_vars))
print("Latitude order:",
      float(ABMsubset.latitude.values[0]),
      "->",
      float(ABMsubset.latitude.values[-1]))
print("GAM rows:", len(gam_curve))


In [ ]:
# Production age-condition lookup
def baseline_class(age):
    return np.interp(
        age,
        gam_curve["Age"].values,
        gam_curve["ExpectedClass"].values,
        left=float(gam_curve["ExpectedClass"].iloc[0]),
        right=float(gam_curve["ExpectedClass"].iloc[-1]),
    )

## Run production deterioration scenarios

In [ ]:
steel_ds = compute_total_decay_full(
    ABMsubset.copy(),
    wreck_material="steel",
    k_teredo=0.0,
    **baseline_params
)

woodnone_ds = compute_total_decay_full(
    ABMsubset.copy(),
    wreck_material="wood",
    k_teredo=0.0,
    **baseline_params
)

woodhigh_ds = compute_total_decay_full(
    ABMsubset.copy(),
    wreck_material="wood",
    k_teredo=teredo_k["high"],
    **baseline_params
)

steel_ref, steel_ref_ref = compute_steel_reference(
    ABMsubset.copy(),
    baseline_params
)

print("Steel reference:", steel_ref_ref)

## Export time-to-Muckelroy-class-3 maps

In [ ]:
def write_north_up_geotiff(da, outfile, dtype=None):
    """
    Write a 2-D latitude/longitude DataArray as a conventional north-up GeoTIFF.

    The publication NetCDF stores latitude south -> north. GeoTIFFs are written
    north -> south so the affine y pixel size is negative and GIS/R readers
    interpret the raster conventionally.
    """
    if "latitude" not in da.dims or "longitude" not in da.dims:
        raise ValueError(
            f"Expected latitude/longitude dimensions, got {da.dims}"
        )

    out = da
    if dtype is not None:
        out = out.astype(dtype)

    # Force longitude west -> east and latitude north -> south.
    out = out.sortby("longitude", ascending=True)
    out = out.sortby("latitude", ascending=False)

    out = (
        out
        .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
        .rio.write_crs("EPSG:4326")
    )

    outfile = Path(outfile)
    outfile.parent.mkdir(parents=True, exist_ok=True)
    out.rio.to_raster(outfile)

    # Immediate sanity check by reopening the file.
    check = rioxarray.open_rasterio(outfile, masked=True)

    transform = check.rio.transform()
    bounds = check.rio.bounds()

    if transform.e >= 0:
        check.close()
        raise RuntimeError(
            f"{outfile.name} was not written north-up: "
            f"y pixel size = {transform.e}"
        )

    if bounds[1] >= bounds[3]:
        check.close()
        raise RuntimeError(
            f"{outfile.name} has invalid south/north bounds: {bounds}"
        )

    check.close()

    return out, transform, bounds


scenarios = {
    "steel_none": {
        "ds": steel_ds,
        "material": "steel",
        "use_teredo": False,
    },
    "wood_none": {
        "ds": woodnone_ds,
        "material": "wood",
        "use_teredo": False,
    },
    "wood_high": {
        "ds": woodhigh_ds,
        "material": "wood",
        "use_teredo": True,
    },
}

for name, sc in scenarios.items():

    tmap = time_to_threshold_map(
        sc["ds"],
        material=sc["material"],
        use_teredo=sc["use_teredo"],
        steel_ref_ref=steel_ref_ref,
        baseline_class=baseline_class,
        threshold=3.0,
        max_years=300,
        alpha=alpha_rate,
    )

    outfile = figure_map_dir / f"{name}_time_to_muckelroy_3.tif"

    written, transform, bounds = write_north_up_geotiff(
        tmap,
        outfile,
        dtype="float32",
    )

    vals = written.values

    print(
        f"{name}: "
        f"{float(np.nanmin(vals)):.1f}–{float(np.nanmax(vals)):.1f} years"
    )
    print("  saved:", outfile)
    print("  y pixel size:", transform.e)
    print("  bounds:", bounds)


## Export environmental and mechanical-forcing driver maps

`fHydro` is retained as the internal variable name for compatibility with `decay_model.py`, but the
manuscript-facing raster is now named **mechanical_forcing_modifier.tif**.

In [ ]:
driver_ds = steel_ds

drivers = {
    "salinity_mean": driver_ds["so_mean"],
    "bottom_temperature_mean": (
        driver_ds["bottomT_mean"]
        if "bottomT_mean" in driver_ds
        else driver_ds["thetao_mean"]
    ),
    "mechanical_forcing_modifier": driver_ds["fHydro"],
    "oxygen_mean": driver_ds["o2_mean"],
    "substrate_hardness": driver_ds["substrate_soft_hard"],
    "CR_physics": driver_ds["CR_physics"],
    "CR_bio": driver_ds["CR_bio"],
}

# Export component forcing terms too when present.
for optional_name in (
    "fCurrent",
    "fWave",
    "current_speed_mean",
    "orbital_velocity",
):
    if optional_name in driver_ds:
        drivers[optional_name] = driver_ds[optional_name]

for name, da in drivers.items():

    outfile = driver_dir / f"{name}.tif"

    _, transform, bounds = write_north_up_geotiff(
        da,
        outfile,
        dtype="float32",
    )

    print("Saved:", outfile)
    print("  y pixel size:", transform.e)
    print("  bounds:", bounds)


## Representative regional trajectories and process fingerprints

In [ ]:
# =====================================================
# Representative deterioration trajectories
# =====================================================

import numpy as np
import pandas as pd

# -----------------------------------------------------
# Representative regional locations
# -----------------------------------------------------
#        "Skagerrak",        58.0,        9.5,

sites = pd.DataFrame({
    "site": [
        "Baltic",
        "Danish Straits",
        "Jutland shelf",
        "Southern North Sea",
        "Norwegian trench",
        "Baltic entrance"
    ],
    "latitude": [
        56.5,
        56.3,
        56.4,
        54.8,
        57.35,
        55
    ],
    "longitude": [
        18.0,
        11.5,
        6.4,
        6.5,
        7.7,
        13
    ]
})

# -----------------------------------------------------
# Scenario metadata
# -----------------------------------------------------

scenario_info = {
    "Steel": {
        "ds": steel_ds,
        "material": "steel",
        "use_teredo": False
    },
    "Wood_none": {
        "ds": woodnone_ds,
        "material": "wood",
        "use_teredo": False
    },
    "Wood_high": {
        "ds": woodhigh_ds,
        "material": "wood",
        "use_teredo": True
    }
}

# -----------------------------------------------------
# Add rate_scale to each dataset
# -----------------------------------------------------

for name, sc in scenario_info.items():

    CR_total, rate_scale = build_rate_scale(
        sc["ds"],
        material=sc["material"],
        use_teredo=sc["use_teredo"],
        steel_ref_ref=steel_ref_ref,
        alpha=alpha_rate
    )

    sc["ds"]["CR_total_for_rate"] = CR_total
    sc["ds"]["rate_scale"] = rate_scale

    print(
        name,
        "rate_scale median =",
        float(np.nanmedian(rate_scale.compute().values))
    )

# -----------------------------------------------------
# Helper: extract 3x3-cell median
# -----------------------------------------------------

def extract_box_median(
    ds,
    lat,
    lon,
    vars_to_extract,
    n_cells=1
):

    lat_vals = ds.latitude.values
    lon_vals = ds.longitude.values

    i = int(np.argmin(np.abs(lat_vals - lat)))
    j = int(np.argmin(np.abs(lon_vals - lon)))

    i0 = max(i - n_cells, 0)
    i1 = min(i + n_cells + 1, len(lat_vals))

    j0 = max(j - n_cells, 0)
    j1 = min(j + n_cells + 1, len(lon_vals))

    sub = ds[vars_to_extract].isel(
        latitude=slice(i0, i1),
        longitude=slice(j0, j1)
    )

    out = {}

    for v in vars_to_extract:
        vals = sub[v].compute().values
        out[v] = float(np.nanmedian(vals))

    out["model_latitude"] = float(lat_vals[i])
    out["model_longitude"] = float(lon_vals[j])

    return out


# =====================================================
# Extract MSM environmental descriptors
# =====================================================

def first_existing_variable(ds, candidates):
    """
    Return the first variable name present in an xarray Dataset.

    Raises a clear error when none of the candidate names exist.
    """
    for variable in candidates:
        if variable in ds:
            return variable

    raise KeyError(
        "None of these variables were found in the dataset: "
        f"{candidates}"
    )


def circular_mean_degrees(values):
    """
    Circular mean of compass bearings in degrees.

    Appropriate for directions such as 359°, 1°, 2°.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    radians = np.deg2rad(values)

    mean_sin = np.mean(np.sin(radians))
    mean_cos = np.mean(np.cos(radians))

    return float(
        np.rad2deg(
            np.arctan2(mean_sin, mean_cos)
        ) % 360.0
    )


def compass_direction_to(u_east, v_north):
    """
    Convert eastward and northward vector components into the
    compass direction towards which the current flows.

    0° = north
    90° = east
    180° = south
    270° = west
    """
    return (
        np.rad2deg(
            np.arctan2(u_east, v_north)
        )
        % 360.0
    )


def extract_reference_seascape(
    ds,
    lat,
    lon,
    n_cells=1
):
    """
    Extract environmental descriptors for one representative site
    using the median of a local 3 × 3 grid-cell window.

    Scalar current speed represents mean hydrodynamic forcing.

    Mean vector speed and direction represent long-term net transport.
    """

    # -------------------------------------------------
    # Resolve actual Navigator variable names
    # -------------------------------------------------

    temperature_var = first_existing_variable(
        ds,
        [
            "bottomT_mean",
            "thetao_bottom_mean",
            "temperature_mean",
            "temperature_C"
        ]
    )

    oxygen_var = first_existing_variable(
        ds,
        [
            "o2_mean",
            "oxygen_mean",
            "bottom_o2_mean",
            "oxygen"
        ]
    )

    salinity_var = first_existing_variable(
        ds,
        [
            "so_mean",
            "salinity_mean",
            "bottom_salinity_mean",
            "salinity"
        ]
    )

    scalar_speed_var = first_existing_variable(
        ds,
        [
            "current_speed_mean",
            "scalar_current_speed",
            "current_speed"
        ]
    )

    u_var = first_existing_variable(
        ds,
        [
            "uo_mean",
            "u_mean",
            "bottom_u_mean"
        ]
    )

    v_var = first_existing_variable(
        ds,
        [
            "vo_mean",
            "v_mean",
            "bottom_v_mean"
        ]
    )

    # -------------------------------------------------
    # Locate grid cell and local window
    # -------------------------------------------------

    lat_vals = ds.latitude.values
    lon_vals = ds.longitude.values

    i = int(
        np.argmin(
            np.abs(lat_vals - lat)
        )
    )

    j = int(
        np.argmin(
            np.abs(lon_vals - lon)
        )
    )

    i0 = max(i - n_cells, 0)
    i1 = min(
        i + n_cells + 1,
        len(lat_vals)
    )

    j0 = max(j - n_cells, 0)
    j1 = min(
        j + n_cells + 1,
        len(lon_vals)
    )

    sub = ds[
        [
            temperature_var,
            oxygen_var,
            salinity_var,
            scalar_speed_var,
            u_var,
            v_var
        ]
    ].isel(
        latitude=slice(i0, i1),
        longitude=slice(j0, j1)
    )

    # -------------------------------------------------
    # Scalar environmental descriptors
    # -------------------------------------------------

    temperature_C = float(
        np.nanmedian(
            sub[temperature_var].compute().values
        )
    )

    oxygen = float(
        np.nanmedian(
            sub[oxygen_var].compute().values
        )
    )

    salinity = float(
        np.nanmedian(
            sub[salinity_var].compute().values
        )
    )

    current_speed = float(
        np.nanmedian(
            sub[scalar_speed_var].compute().values
        )
    )

    # -------------------------------------------------
    # Mean current vector
    #
    # Take local median components first, then calculate
    # vector speed and direction from those components.
    # -------------------------------------------------

    u_mean = float(
        np.nanmedian(
            sub[u_var].compute().values
        )
    )

    v_mean = float(
        np.nanmedian(
            sub[v_var].compute().values
        )
    )

    current_vector_speed = float(
        np.hypot(
            u_mean,
            v_mean
        )
    )

    current_direction_to_deg = float(
        compass_direction_to(
            u_east=u_mean,
            v_north=v_mean
        )
    )

    if (
        np.isfinite(current_speed)
        and current_speed > 0.0
    ):
        directional_persistence = float(
            np.clip(
                current_vector_speed
                / current_speed,
                0.0,
                1.0
            )
        )
    else:
        directional_persistence = np.nan

    return {
        "model_latitude": float(
            lat_vals[i]
        ),
        "model_longitude": float(
            lon_vals[j]
        ),
        "temperature_C": temperature_C,
        "oxygen": oxygen,
        "salinity": salinity,
        "current_speed": current_speed,
        "current_vector_speed": current_vector_speed,
        "current_direction_to_deg": (
            current_direction_to_deg
        ),
        "directional_persistence": (
            directional_persistence
        )
    }


# -----------------------------------------------------
# Extract the six Heritage Science seascapes
# -----------------------------------------------------

reference_rows = []

# Environmental descriptors do not depend on material
# scenario, so use one authoritative Navigator dataset.
environment_ds = steel_ds

for _, site in sites.iterrows():

    values = extract_reference_seascape(
        ds=environment_ds,
        lat=site["latitude"],
        lon=site["longitude"],
        n_cells=1
    )

    reference_rows.append({
        "name": site["site"],
        "latitude": (
            site["latitude"]
        ),
        "longitude": (
            site["longitude"]
        ),
        "model_latitude": (
            values["model_latitude"]
        ),
        "model_longitude": (
            values["model_longitude"]
        ),
        "temperature_C": (
            values["temperature_C"]
        ),
        "oxygen": (
            values["oxygen"]
        ),
        "salinity": (
            values["salinity"]
        ),
        "current_speed": (
            values["current_speed"]
        ),
        "current_vector_speed": (
            values["current_vector_speed"]
        ),
        "current_direction_to_deg": (
            values["current_direction_to_deg"]
        ),
        "directional_persistence": (
            values["directional_persistence"]
        ),

        # MSM-specific values remain undefined until
        # their physical formulations are agreed.
        "storm_scale": np.nan,
        "sediment_mobility": np.nan,
        "deposition_bias": np.nan,
        "substrate": ""
    })


reference_seascapes = pd.DataFrame(
    reference_rows
)

# Match the order expected by the modular MSM package.
reference_seascapes = reference_seascapes[
    [
        "name",
        "latitude",
        "longitude",
        "model_latitude",
        "model_longitude",
        "temperature_C",
        "oxygen",
        "salinity",
        "current_speed",
        "current_vector_speed",
        "current_direction_to_deg",
        "directional_persistence",
        "storm_scale",
        "sediment_mobility",
        "deposition_bias",
        "substrate"
    ]
]

print(reference_seascapes.to_string(index=False))

reference_outfile = os.path.join(
    project_root,
    "reference_seascapes.csv"
)

reference_seascapes.to_csv(
    reference_outfile,
    index=False
)

print(
    "Saved:",
    reference_outfile
)



# -----------------------------------------------------
# Extract representative site rates
# -----------------------------------------------------

site_rates = []

for _, site in sites.iterrows():

    for scenario, sc in scenario_info.items():

        vals = extract_box_median(
            sc["ds"],
            lat=site["latitude"],
            lon=site["longitude"],
            vars_to_extract=["rate_scale"],
            n_cells=1
        )

        site_rates.append({
            "site": site["site"],
            "scenario": scenario,
            "target_latitude": site["latitude"],
            "target_longitude": site["longitude"],
            "model_latitude": vals["model_latitude"],
            "model_longitude": vals["model_longitude"],
            "rate_scale": vals["rate_scale"]
        })

site_rates = pd.DataFrame(site_rates)

print(site_rates)

# -----------------------------------------------------
# Load calibrated GAM curve
# -----------------------------------------------------

gam_curve = pd.read_csv(
    os.path.join(
        project_root,
        "data",
        "gam_age_curve.csv"
    )
)

# -----------------------------------------------------
# Build trajectories
# -----------------------------------------------------

age_years = np.arange(0, 301)

trajectory_rows = []

for _, row in site_rates.iterrows():

    effective_age = age_years * row["rate_scale"]

    predicted_condition = np.interp(
        effective_age,
        gam_curve["Age"],
        gam_curve["ExpectedClass"],
        left=gam_curve["ExpectedClass"].iloc[0],
        right=gam_curve["ExpectedClass"].iloc[-1]
    )

    for age, eff_age, cond in zip(
        age_years,
        effective_age,
        predicted_condition
    ):

        trajectory_rows.append({
            "site": row["site"],
            "scenario": row["scenario"],
            "chronological_age": age,
            "effective_age": eff_age,
            "predicted_condition": cond
        })

trajectory_df = pd.DataFrame(
    trajectory_rows
)

print(trajectory_df.head())

# -----------------------------------------------------
# Export
# -----------------------------------------------------

outfile = os.path.join(
    project_root,
    "representative_trajectories.csv"
)

trajectory_df.to_csv(
    outfile,
    index=False
)

print("Saved:", outfile)


In [ ]:
# -----------------------------------------------------
# Extract representative site process contributions
# -----------------------------------------------------

process_vars = [
    "CR_physics",
    "CR_bio",
    "CR_teredo",
    "CR_total_for_rate",
    "rate_scale"
]

process_rows = []

for _, site in sites.iterrows():

    for scenario, sc in scenario_info.items():

        vals = extract_box_median(
            sc["ds"],
            lat=site["latitude"],
            lon=site["longitude"],
            vars_to_extract=process_vars,
            n_cells=1
        )

        CR_physics = vals["CR_physics"]
        CR_bio = vals["CR_bio"]
        CR_teredo = vals["CR_teredo"]
        CR_total = vals["CR_total_for_rate"]

        if CR_total > 0:
            physics_pct = 100 * CR_physics / CR_total
            bio_pct = 100 * CR_bio / CR_total
            teredo_pct = 100 * CR_teredo / CR_total
        else:
            physics_pct = np.nan
            bio_pct = np.nan
            teredo_pct = np.nan

        process_rows.append({
            "site": site["site"],
            "scenario": scenario,
            "target_latitude": site["latitude"],
            "target_longitude": site["longitude"],
            "model_latitude": vals["model_latitude"],
            "model_longitude": vals["model_longitude"],
            "CR_physics": CR_physics,
            "CR_bio": CR_bio,
            "CR_teredo": CR_teredo,
            "CR_total": CR_total,
            "rate_scale": vals["rate_scale"],
            "physics_pct": physics_pct,
            "bio_pct": bio_pct,
            "teredo_pct": teredo_pct
        })

process_df = pd.DataFrame(process_rows)

print(process_df)

outfile_process = figure_data_dir / "representative_process_contributions.csv"

process_df.to_csv(outfile_process, index=False)

print("Saved:", outfile_process)


In [ ]:
# Compact representative-site rate table used to construct trajectories
site_rate_rows = []

for _, site in sites.iterrows():
    for scenario, sc in scenario_info.items():
        vals = extract_box_median(
            sc["ds"],
            lat=site["latitude"],
            lon=site["longitude"],
            vars_to_extract=["rate_scale"],
            n_cells=1,
        )
        site_rate_rows.append({
            "site": site["site"],
            "scenario": scenario,
            "rate_scale": vals["rate_scale"],
        })

site_rates = pd.DataFrame(site_rate_rows)
site_rates

In [ ]:
# gam_curve already loaded from gam_age_curve.csv:
# columns: Age, ExpectedClass

age_years = np.arange(0, 301)

def class_from_effective_age(eff_age, gam_curve):
    return np.interp(
        eff_age,
        gam_curve["Age"].values,
        gam_curve["ExpectedClass"].values,
        left=gam_curve["ExpectedClass"].iloc[0],
        right=gam_curve["ExpectedClass"].iloc[-1]
    )

traj_rows = []

for _, row in site_rates.iterrows():
    eff_age = age_years * row["rate_scale"]
    condition = class_from_effective_age(eff_age, gam_curve)

    for age, cond in zip(age_years, condition):
        traj_rows.append({
            "site": row["site"],
            "scenario": row["scenario"],
            "chronological_age": age,
            "effective_age": age * row["rate_scale"],
            "predicted_condition": cond,
            "rate_scale": row["rate_scale"]
        })

trajectory_df = pd.DataFrame(traj_rows)

outfile = figure_data_dir / "representative_site_trajectories.csv"
outfile.parent.mkdir(parents=True, exist_ok=True)

trajectory_df.to_csv(outfile, index=False)
print("Saved:", outfile)


## Optional spatial sample for supplementary variance plots

In [ ]:
import numpy as np
import pandas as pd

def export_spatial_sample(
    ds,
    vars_to_export,
    outfile,
    sample=10000,
    seed=123,
    require_all=False
):
    """
    Export a tractable lat/lon cell sample from an xarray Dataset.

    Each row = one unique spatial cell.
    Keeps latitude and longitude.

    require_all = False:
        keep cells with at least one finite requested variable.
    require_all = True:
        keep only cells with finite values for all requested variables.
    """

    # keep only variables that exist
    vars_present = [v for v in vars_to_export if v in ds.data_vars]
    missing = [v for v in vars_to_export if v not in ds.data_vars]

    if missing:
        print("Missing variables skipped:")
        for v in missing:
            print("  ", v)

    if len(vars_present) == 0:
        raise ValueError("None of the requested variables were found in dataset.")

    # subset and load only requested variables
    sub = ds[vars_present].compute()

    # convert to long table with one row per lat/lon cell
    df = sub.to_dataframe().reset_index()

    # keep only lat/lon + requested variables
    df = df[["latitude", "longitude"] + vars_present]

    # remove land / empty cells
    if require_all:
        df = df.dropna(subset=vars_present, how="any")
    else:
        df = df.dropna(subset=vars_present, how="all")

    # sample unique cells
    if sample is not None and len(df) > sample:
        df = df.sample(
            n=sample,
            random_state=seed
        ).sort_values(["latitude", "longitude"])

    df.to_csv(outfile, index=False)

    print(f"Exported {len(df)} rows")
    print(f"Variables exported: {len(vars_present)}")
    print(outfile)

    return df

In [ ]:
vars_to_export = [
    "bottomT_mean",
    "bottomT_var_total",
#    "bottomT_seasonal_amp",
#    "bottomT_var_interannual",

#   "o2_mean",
#   "o2_var_total",
#    "o2_seasonal_amp",
#   "o2_var_interannual",

    "so_mean",
    "so_var_total",
#    "so_seasonal_amp",
    "so_var_interannual",

    "chl_mean",
    "chl_var_total",
    "chl_var_interannual",
#    "nppv_mean",
#    "nppv_var_total",
#    "phyc_mean",
#    "phyc_var_total",

    "VHM0_mean_mean",
    "VHM0_mean_var_total",

    "VHM0_p95_mean"
#    "fishing_bottom_contact_mean",
#    "fishing_bottom_contact_var_index"
]

outfile = figure_data_dir / "variance_spatial_sample.csv"

variance_sample_df = export_spatial_sample(
    ABMsubset,
    vars_to_export=vars_to_export,
    outfile=outfile,
    sample=10000,
    seed=42,
    require_all=False
)


## Outputs

The principal products are written beneath `outputs/figure_maps`, `outputs/driver_maps` and
`outputs/figure_data`. Notebook 10 should only read these products and plot them; it should not
rerun the deterioration model.